# Bài tập vận dụng: RNN dự đoán từ tiếp theo

Notebook này xây dựng một mô hình **RNN Language Model** đơn giản để dự đoán từ tiếp theo.

Chủ đề văn bản: **nước và vòng tuần hoàn nước**.

Các yêu cầu chính trong bài:

1. Tiền xử lý văn bản: xây dựng từ điển và chuyển từ thành số.
2. Xây dựng mô hình gồm `nn.Embedding`, `nn.RNN`, `nn.Linear`.
3. Áp dụng **Weight Tying** bằng cách dùng chung trọng số giữa `Embedding` và `Linear`.
4. Huấn luyện mô hình bằng `CrossEntropyLoss`.
5. Áp dụng **Teacher Forcing** trong quá trình tạo dữ liệu huấn luyện.

In [ ]:
import re
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

thiet_bi = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị đang dùng:", thiet_bi)

Thiết bị đang dùng: cuda


In [ ]:
van_ban = """
nuoc la mot tai nguyen quan trong doi voi su song tren trai dat
nuoc co the ton tai o ba trang thai la ran long va khi
mat troi lam nuoc o song ho va bien bay hoi len cao
hoi nuoc gap lanh se ngung tu thanh may tren bau troi
khi may chua nhieu hoi nuoc chung co the tao ra mua
nuoc mua roi xuong mat dat va tham vao long dat
mot phan nuoc chay ve song suoi ao ho va bien
cay coi dong vat va con nguoi deu can nuoc de song
vong tuan hoan nuoc giup duy tri su can bang trong tu nhien
khoa hoc giup con nguoi hieu ro vai tro cua nuoc doi voi trai dat
"""

print(van_ban)


nuoc la mot tai nguyen quan trong doi voi su song tren trai dat
nuoc co the ton tai o ba trang thai la ran long va khi
mat troi lam nuoc o song ho va bien bay hoi len cao
hoi nuoc gap lanh se ngung tu thanh may tren bau troi
khi may chua nhieu hoi nuoc chung co the tao ra mua
nuoc mua roi xuong mat dat va tham vao long dat
mot phan nuoc chay ve song suoi ao ho va bien
cay coi dong vat va con nguoi deu can nuoc de song
vong tuan hoan nuoc giup duy tri su can bang trong tu nhien
khoa hoc giup con nguoi hieu ro vai tro cua nuoc doi voi trai dat



## Tiền xử lý văn bản

- Đưa văn bản về chữ thường.
- Loại bỏ ký tự không cần thiết.
- Tách văn bản thành danh sách các từ.
- Xây dựng từ điển `tu_sang_so` để đổi từ thành số.
- Xây dựng từ điển `so_sang_tu` để đổi số ngược lại thành từ.
- Chuyển toàn bộ văn bản thành một chuỗi số.

In [ ]:
def tien_xu_ly_van_ban(van_ban):
    van_ban = van_ban.lower()
    van_ban = re.sub(r"[^a-zA-Z0-9\s]", " ", van_ban)
    danh_sach_tu = van_ban.split()
    return danh_sach_tu


danh_sach_tu = tien_xu_ly_van_ban(van_ban)

tu_khong_trung = sorted(set(danh_sach_tu))

tu_sang_so = {tu: chi_so for chi_so, tu in enumerate(tu_khong_trung)}
so_sang_tu = {chi_so: tu for tu, chi_so in tu_sang_so.items()}

chuoi_so = [tu_sang_so[tu] for tu in danh_sach_tu]
kich_thuoc_tu_dien = len(tu_sang_so)

print("Số lượng từ trong văn bản:", len(danh_sach_tu))
print("Kích thước từ điển:", kich_thuoc_tu_dien)

print("\nMột vài từ trong từ điển:")
print(list(tu_sang_so.items())[:15])

print("\nVăn bản sau khi chuyển thành số:")
print(chuoi_so[:30])

Số lượng từ trong văn bản: 127
Kích thước từ điển: 81

Một vài từ trong từ điển:
[('ao', 0), ('ba', 1), ('bang', 2), ('bau', 3), ('bay', 4), ('bien', 5), ('can', 6), ('cao', 7), ('cay', 8), ('chay', 9), ('chua', 10), ('chung', 11), ('co', 12), ('coi', 13), ('con', 14)]

Văn bản sau khi chuyển thành số:
[45, 31, 38, 57, 42, 48, 70, 19, 78, 55, 54, 66, 64, 16, 45, 12, 62, 63, 57, 46, 1, 65, 59, 31, 50, 35, 73, 29, 36, 69]


## Tạo dữ liệu huấn luyện bằng Teacher Forcing

Với mô hình dự đoán từ tiếp theo, ta tạo dữ liệu huấn luyện bằng cách lấy một chuỗi từ làm đầu vào và dịch chuỗi đó sang phải 1 từ để làm nhãn.

Ví dụ chuỗi từ trong văn bản:

```text
nuoc la mot tai nguyen quan trong
```

Dữ liệu vào:

```text
nuoc la mot tai nguyen
```

Nhãn cần dự đoán:

```text
la mot tai nguyen quan
```

Nghĩa là tại mỗi bước thời gian, mô hình nhận **từ thật hiện tại** để dự đoán **từ thật kế tiếp**.

Ví dụ:

- Khi đầu vào là `nuoc`, mô hình cần dự đoán `la`.
- Khi đầu vào là `la`, mô hình cần dự đoán `mot`.
- Khi đầu vào là `mot`, mô hình cần dự đoán `tai`.


In [ ]:
def tao_du_lieu_huan_luyen(chuoi_so, do_dai_ngu_canh):
    danh_sach_dau_vao = []
    danh_sach_nhan = []

    for vi_tri in range(len(chuoi_so) - do_dai_ngu_canh):
        dau_vao = chuoi_so[vi_tri : vi_tri + do_dai_ngu_canh]
        nhan = chuoi_so[vi_tri + 1 : vi_tri + do_dai_ngu_canh + 1]

        danh_sach_dau_vao.append(dau_vao)
        danh_sach_nhan.append(nhan)

    return torch.tensor(danh_sach_dau_vao), torch.tensor(danh_sach_nhan)


do_dai_ngu_canh = 5

X, y = tao_du_lieu_huan_luyen(chuoi_so, do_dai_ngu_canh)

bo_du_lieu = TensorDataset(X, y)
bo_tai_du_lieu = DataLoader(bo_du_lieu, batch_size=4, shuffle=True)


##Xây dựng mô hình RNN

Mô hình gồm 3 phần chính:

1. `nn.Embedding`: chuyển mỗi từ từ dạng số sang vector.
2. `nn.RNN`: xử lý chuỗi vector theo từng bước thời gian.
3. `nn.Linear`: chuyển đầu ra của RNN thành điểm dự đoán cho từng từ trong từ điển.

Trong bài này, ta áp dụng **Weight Tying**:

```python
self.fc.weight = self.embedding.weight
```

In [ ]:
class MoHinhRNN(nn.Module):
    def __init__(self, kich_thuoc_tu_dien, kich_thuoc_embedding, kich_thuoc_an):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=kich_thuoc_tu_dien,
            embedding_dim=kich_thuoc_embedding
        )

        self.rnn = nn.RNN(
            input_size=kich_thuoc_embedding,
            hidden_size=kich_thuoc_an,
            batch_first=True
        )

        self.fc = nn.Linear(
            in_features=kich_thuoc_an,
            out_features=kich_thuoc_tu_dien
        )

        # Weight Tying: dùng chung trọng số giữa Embedding và Linear đầu ra
        self.fc.weight = self.embedding.weight

    def forward(self, dau_vao):
        vector_tu = self.embedding(dau_vao)
        dau_ra_rnn, trang_thai_an = self.rnn(vector_tu)
        du_doan = self.fc(dau_ra_rnn)

        return du_doan


kich_thuoc_embedding = 64
kich_thuoc_an = 64

mo_hinh = MoHinhRNN(
    kich_thuoc_tu_dien=kich_thuoc_tu_dien,
    kich_thuoc_embedding=kich_thuoc_embedding,
    kich_thuoc_an=kich_thuoc_an
).to(thiet_bi)

print(mo_hinh)

print("\nKiểm tra Weight Tying:")
print("fc.weight có dùng chung với embedding.weight không?")
print(mo_hinh.fc.weight is mo_hinh.embedding.weight)

MoHinhRNN(
  (embedding): Embedding(81, 64)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=81, bias=True)
)

Kiểm tra Weight Tying:
fc.weight có dùng chung với embedding.weight không?
True


## Huấn luyện mô hình

Ta sử dụng:

- `CrossEntropyLoss`: hàm mất mát cho bài toán dự đoán từ tiếp theo.
- `Adam`: thuật toán tối ưu trọng số.

Kích thước dữ liệu khi huấn luyện:

- `du_doan`: có dạng `(batch_size, do_dai_ngu_canh, kich_thuoc_tu_dien)`.
- `nhan`: có dạng `(batch_size, do_dai_ngu_canh)`.

Trước khi đưa vào `CrossEntropyLoss`, ta trải phẳng hai tensor này.

In [ ]:
ham_mat_mat = nn.CrossEntropyLoss()
bo_toi_uu = torch.optim.Adam(mo_hinh.parameters(), lr=0.01)

so_epoch = 300

for epoch in range(1, so_epoch + 1):
    tong_loss = 0

    for dau_vao, nhan in bo_tai_du_lieu:
        dau_vao = dau_vao.to(thiet_bi)
        nhan = nhan.to(thiet_bi)

        du_doan = mo_hinh(dau_vao)

        loss = ham_mat_mat(
            du_doan.reshape(-1, kich_thuoc_tu_dien),
            nhan.reshape(-1)
        )

        bo_toi_uu.zero_grad()
        loss.backward()
        bo_toi_uu.step()

        tong_loss += loss.item()

    if epoch % 50 == 0:
        loss_trung_binh = tong_loss / len(bo_tai_du_lieu)
        print(f"Epoch {epoch:3d} | Loss trung bình: {loss_trung_binh:.4f}")

Epoch  50 | Loss trung bình: 0.2068
Epoch 100 | Loss trung bình: 0.1868
Epoch 150 | Loss trung bình: 0.2039
Epoch 200 | Loss trung bình: 0.2070
Epoch 250 | Loss trung bình: 0.2142
Epoch 300 | Loss trung bình: 0.2196


## Hàm dự đoán từ tiếp theo

Hàm dưới đây nhận vào một đoạn ngữ cảnh ngắn, sau đó dự đoán từ có điểm cao nhất là từ tiếp theo.

In [ ]:
def du_doan_tu_tiep_theo(mo_hinh, cau_dau_vao):
    mo_hinh.eval()

    danh_sach_tu_vao = tien_xu_ly_van_ban(cau_dau_vao)

    danh_sach_so = []
    for tu in danh_sach_tu_vao:
        if tu in tu_sang_so:
            danh_sach_so.append(tu_sang_so[tu])

    if len(danh_sach_so) == 0:
        return "Không có từ nào trong câu nằm trong từ điển."

    # Chỉ lấy tối đa do_dai_ngu_canh từ cuối cùng
    danh_sach_so = danh_sach_so[-do_dai_ngu_canh:]

    dau_vao = torch.tensor([danh_sach_so]).to(thiet_bi)

    with torch.no_grad():
        du_doan = mo_hinh(dau_vao)

        # Lấy dự đoán tại bước thời gian cuối cùng
        diem_tu_cuoi = du_doan[0, -1]
        chi_so_tu_du_doan = torch.argmax(diem_tu_cuoi).item()

    return so_sang_tu[chi_so_tu_du_doan]


cau_thu = "nuoc la mot tai"
tu_tiep_theo = du_doan_tu_tiep_theo(mo_hinh, cau_thu)

print("Câu đầu vào:", cau_thu)
print("Từ tiếp theo mô hình dự đoán:", tu_tiep_theo)

Câu đầu vào: nuoc la mot tai
Từ tiếp theo mô hình dự đoán: nguyen


## Sinh thêm nhiều từ liên tiếp

Ngoài dự đoán một từ tiếp theo, ta có thể cho mô hình sinh tiếp nhiều từ.

Sau mỗi lần dự đoán, từ vừa dự đoán sẽ được thêm vào câu hiện tại.  
Sau đó, mô hình tiếp tục dùng câu mới để dự đoán từ kế tiếp.

In [ ]:
def sinh_van_ban(mo_hinh, cau_bat_dau, so_tu_can_sinh):
    cau_hien_tai = cau_bat_dau

    for _ in range(so_tu_can_sinh):
        tu_moi = du_doan_tu_tiep_theo(mo_hinh, cau_hien_tai)

        if tu_moi.startswith("Không có"):
            break

        cau_hien_tai = cau_hien_tai + " " + tu_moi

    return cau_hien_tai


cau_bat_dau = "nuoc la mot"
ket_qua = sinh_van_ban(mo_hinh, cau_bat_dau, so_tu_can_sinh=10)

print("Câu bắt đầu:", cau_bat_dau)
print("Văn bản sinh ra:")
print(ket_qua)

Câu bắt đầu: nuoc la mot
Văn bản sinh ra:
nuoc la mot tai nguyen quan trong doi voi su song tren trai
